# UCI Air Quality — Regression & Classification Models

**Project Deliverable 2 | Python analytics and model demonstration**

This notebook uses the **cleaned UCI Air Quality dataset from Deliverable 1**. It builds:

1. **Regression:** predict numeric **NO2(GT)** from sensor and weather measurements.
2. **Classification:** predict whether NO2(GT) is **at or above the median of the training observations**. This is a demonstration category, **not an official air-quality standard or health warning**.

**What you will produce:** trained/evaluated model results, predictions CSV, three PNG charts, and a written results summary. Upload these outputs along with the notebook and README to your GitHub repository.

**Important:** This notebook does not contain invented model scores. It calculates metrics only after you run it on your actual cleaned CSV.


## 1 — Upload your cleaned CSV

In Colab, select **Runtime → Run all**. When prompted, upload **`AirQualityUCI_clean.csv`** (the *cleaned* file, not the original semicolon-delimited source). You already downloaded this in Deliverable 1.


In [ ]:
from pathlib import Path
import json
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)

DATA_FILE = Path('AirQualityUCI_clean.csv')
OUTPUT_DIR = Path('model_outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

if not DATA_FILE.exists():
    from google.colab import files
    print('Please choose AirQualityUCI_clean.csv from your Downloads folder.')
    files.upload()

if not DATA_FILE.exists():
    raise FileNotFoundError('AirQualityUCI_clean.csv was not uploaded. Run this cell again and select the cleaned CSV.')

df = pd.read_csv(DATA_FILE)
print(f'Loaded: {DATA_FILE} | {len(df):,} rows | {len(df.columns)} columns')
print('Columns:', ', '.join(df.columns))
print(df.head(3).to_string(index=False))


## 2 — Prepare the dataset carefully

We use the five metal-oxide sensor channels plus three weather measurements as input features. The model targets **NO2(GT)** (nitrogen dioxide ground-truth measurements). We **do not** include the target itself, or **NOx(GT)**, among predictors.

- Missing target values are **excluded** from supervised training/testing; they are not invented.
- Missing input features are replaced with medians **learned only from training data**.
- Records are ordered chronologically, with the first 80% used for training and the last 20% for testing, to assess performance on later observations.


In [ ]:
FEATURES = [
    'PT08.S1(CO)', 'PT08.S2(NMHC)', 'PT08.S3(NOx)',
    'PT08.S4(NO2)', 'PT08.S5(O3)', 'T', 'RH', 'AH'
]
TARGET = 'NO2(GT)'
required = ['DateTime', TARGET] + FEATURES
missing_cols = [c for c in required if c not in df.columns]
if missing_cols:
    raise ValueError(f'Missing expected columns: {missing_cols}. Upload the CLEANED CSV from Deliverable 1.')

data = df[required].copy()
data['DateTime'] = pd.to_datetime(data['DateTime'], errors='coerce')
for name in FEATURES + [TARGET]:
    data[name] = pd.to_numeric(data[name], errors='coerce')
    data[name] = data[name].replace(-200, np.nan)  # safeguards if a sentinel remains

original_count = len(data)
data = data.dropna(subset=['DateTime', TARGET]).sort_values('DateTime').reset_index(drop=True)
if len(data) < 100 or data[TARGET].nunique() < 3:
    raise ValueError('Too few usable observations; verify that you uploaded the complete cleaned UCI CSV.')

split_index = int(len(data) * 0.8)
train = data.iloc[:split_index].copy()
test = data.iloc[split_index:].copy()
X_train, X_test = train[FEATURES], test[FEATURES]
y_train, y_test = train[TARGET], test[TARGET]

threshold = float(y_train.median())
y_train_class = (y_train >= threshold).astype(int)
y_test_class = (y_test >= threshold).astype(int)

if y_train_class.nunique() != 2:
    raise ValueError('Training data lacks both classification categories.')

print(f'Rows in cleaned file: {original_count:,}')
print(f'Usable NO2 observations: {len(data):,}')
print(f'Dropped missing target or date: {original_count-len(data):,}')
print(f'Training rows: {len(train):,} | Testing rows: {len(test):,}')
print(f'Training median NO2 threshold: {threshold:.2f} (data units)')
print(f'Train period: {train.DateTime.min()} through {train.DateTime.max()}')
print(f'Test period:  {test.DateTime.min()} through {test.DateTime.max()}')


## 3 — Train two models

We choose **Random Forests** as clear, established examples of supervised learning. Both pipelines impute missing predictors using training medians. Reproducibility is supported by a fixed random seed (`random_state=42`).


In [ ]:
regression_model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('model', RandomForestRegressor(
        n_estimators=120, min_samples_leaf=3, random_state=42, n_jobs=-1
    ))
])
classification_model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('model', RandomForestClassifier(
        n_estimators=120, min_samples_leaf=3, random_state=42, n_jobs=-1
    ))
])

regression_model.fit(X_train, y_train)
classification_model.fit(X_train, y_train_class)
reg_pred = regression_model.predict(X_test)
class_pred = classification_model.predict(X_test)
print('Both models trained and evaluated on a later, held-out test period.')


## 4 — Measure model performance

**Regression:** Mean Absolute Error (MAE), Root Mean Squared Error (RMSE), and R².

**Classification:** Accuracy, precision, recall, F1 score, and a confusion matrix.

Interpret results as performance on *this historical dataset's holdout period*, not as proof of general effectiveness for other locations or years.


In [ ]:
reg_metrics = {
    'MAE': float(mean_absolute_error(y_test, reg_pred)),
    'RMSE': float(np.sqrt(mean_squared_error(y_test, reg_pred))),
    'R2': float(r2_score(y_test, reg_pred)),
}
class_metrics = {
    'Accuracy': float(accuracy_score(y_test_class, class_pred)),
    'Precision': float(precision_score(y_test_class, class_pred, zero_division=0)),
    'Recall': float(recall_score(y_test_class, class_pred, zero_division=0)),
    'F1': float(f1_score(y_test_class, class_pred, zero_division=0)),
}
cm = confusion_matrix(y_test_class, class_pred, labels=[0, 1])

print('REGRESSION — Predict NO2(GT)')
for key, value in reg_metrics.items():
    print(f'  {key:>9}: {value:.4f}')
print('\nCLASSIFICATION — NO2 below / at-or-above training median')
for key, value in class_metrics.items():
    print(f'  {key:>9}: {value:.4f}')
print('\nConfusion matrix (actual rows, predicted columns):')
print(cm)
print('\n', classification_report(y_test_class, class_pred,
    labels=[0, 1], target_names=['Below median','At/above median'], zero_division=0))


## 5 — Save graphs and individual predictions

The graphs make the outputs easy to explain in a class report or GitHub repository.


In [ ]:
# Graph A: actual and predicted NO2 across the first 250 holdout hours
shown = min(250, len(test))
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(test['DateTime'].iloc[:shown], y_test.iloc[:shown], label='Actual NO2', linewidth=1.6)
ax.plot(test['DateTime'].iloc[:shown], reg_pred[:shown], label='Predicted NO2', linewidth=1.4)
ax.set(title='Regression: actual vs predicted NO2 (first test observations)',
       xlabel='Date and time', ylabel='NO2(GT) value')
ax.legend()
ax.grid(alpha=0.2)
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'regression_actual_vs_predicted.png', dpi=160)
plt.show()

# Graph B: confusion matrix
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm, display_labels=['Below median', 'At/above median']).plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Classification: held-out confusion matrix')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'classification_confusion_matrix.png', dpi=160)
plt.show()

# Graph C: regression input feature importance
feature_importances = pd.Series(
    regression_model.named_steps['model'].feature_importances_, index=FEATURES
).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(9, 5))
feature_importances.plot.barh(ax=ax)
ax.set(title='Random Forest regression feature importance', xlabel='Relative model importance', ylabel='Input feature')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'regression_feature_importance.png', dpi=160)
plt.show()

results = pd.DataFrame({
    'DateTime': test['DateTime'].dt.strftime('%Y-%m-%d %H:%M:%S').to_numpy(),
    'Actual_NO2': y_test.to_numpy(),
    'Predicted_NO2': np.round(reg_pred, 3),
    'Actual_HigherNO2_Class': y_test_class.to_numpy(),
    'Predicted_HigherNO2_Class': class_pred,
})
results.to_csv(OUTPUT_DIR / 'model_predictions.csv', index=False)
print(f'Created 3 graphs and model_predictions.csv in {OUTPUT_DIR}/')


## 6 — Generate results documentation

This Markdown summary is generated using the actual model metrics, sample counts, and dates from your Colab execution. Add it to GitHub alongside the notebook and outputs.


In [ ]:
metrics = {
    'model_target': TARGET,
    'classification_threshold_training_median': threshold,
    'rows_original_clean_csv': original_count,
    'rows_used': len(data),
    'rows_train': len(train),
    'rows_test': len(test),
    'regression': reg_metrics,
    'classification': class_metrics,
    'confusion_matrix_actual_by_predicted': cm.tolist(),
    'features': FEATURES,
    'train_period': [str(train.DateTime.min()), str(train.DateTime.max())],
    'test_period': [str(test.DateTime.min()), str(test.DateTime.max())],
}
(OUTPUT_DIR / 'model_metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')

summary = f'''# Model results — UCI Air Quality

## Project purpose
Analyze the cleaned historical UCI Air Quality dataset with a numeric prediction model (regression)
and a binary category model (classification).

## Dataset and approach
- Source: https://archive.ics.uci.edu/dataset/360/air+quality
- Input file: `AirQualityUCI_clean.csv`
- Rows in cleaned input: {original_count:,}
- Rows usable after excluding missing targets/dates: {len(data):,}
- Chronological split: {len(train):,} train / {len(test):,} test (first 80% / last 20%)
- Training dates: {train.DateTime.min()} to {train.DateTime.max()}
- Testing dates: {test.DateTime.min()} to {test.DateTime.max()}
- Predictors: {', '.join(FEATURES)}
- Missing feature values filled using medians learned from training only
- Models: RandomForestRegressor and RandomForestClassifier, 120 trees each

## Regression (predict numerical NO2(GT))
- Mean Absolute Error (MAE): {reg_metrics['MAE']:.4f}
- Root Mean Squared Error (RMSE): {reg_metrics['RMSE']:.4f}
- R²: {reg_metrics['R2']:.4f}

## Classification (categorize NO2(GT))
- Category 1: at or above training median ({threshold:.2f} dataset units)
- Category 0: below training median
- Accuracy: {class_metrics['Accuracy']:.4f}
- Precision: {class_metrics['Precision']:.4f}
- Recall: {class_metrics['Recall']:.4f}
- F1: {class_metrics['F1']:.4f}
- Confusion matrix, actual rows / predicted columns: {cm.tolist()}

## Generated outputs
- `regression_actual_vs_predicted.png`: comparison of observed/predicted test values
- `classification_confusion_matrix.png`: correct versus incorrect category predictions
- `regression_feature_importance.png`: model's relative predictor importance
- `model_predictions.csv`: each held-out observation's actual and predicted values/classes
- `model_metrics.json`: machine-readable score summary

## Interpretation and limitations
The regression errors describe deviations between predicted and observed NO2 measurements.
Classification metrics describe agreement with a category defined from **the training median**.
This category is for demonstration only, not a regulatory air-quality or health threshold.
The dataset comes from one historical monitoring location, so performance may not generalize to
other places, periods, or sensors. Some predictors and targets are missing; exclusion and
imputation can affect the evaluation. Random Forest feature importance is not evidence of causality.
The chronological holdout tests later observations but does not replace external validation.

## Source citation
Vito, S. (2008). *Air Quality* [Dataset]. UCI Machine Learning Repository.
https://doi.org/10.24432/C59K5F (CC BY 4.0).
'''
(OUTPUT_DIR / 'MODEL_RESULTS.md').write_text(summary, encoding='utf-8')
print(summary)


## 7 — Download outputs for GitHub

The next cell packages the results into one ZIP for convenient download. **Unzip it on your computer before uploading the individual files into GitHub.** The original cleaned CSV and this `.ipynb` notebook should also be included in your repository.


In [ ]:
archive_path = Path('UCI_Air_Quality_Model_Results.zip')
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for item in sorted(OUTPUT_DIR.glob('*')):
        archive.write(item, arcname=f'model_outputs/{item.name}')
print(f'Packed {len(list(OUTPUT_DIR.glob("*")))} output files into {archive_path}')

try:
    from google.colab import files
    files.download(str(archive_path))
except ImportError:
    print('Not in Colab; ZIP is available in the current working directory.')
